<a href="https://colab.research.google.com/github/royangan/Apollo/blob/test/BBox_StrucutreRecognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 50.5 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF
import sys
from typing import List, Dict
from difflib import SequenceMatcher


In [3]:
def extract_structure(pdf_path: str) -> List[List[Dict]]:
    doc = fitz.open(pdf_path)
    structure_data = []

    for page in doc:
        blocks = page.get_text("dict")["blocks"]
        page_structure = []

        for block in blocks:
            if "lines" in block:
                for line in block["lines"]:
                    line_text = ""
                    bbox = None
                    size = None
                    font = None

                    for span in line["spans"]:
                        line_text += span["text"].strip()
                        bbox = span["bbox"]
                        size = span["size"]
                        font = span["font"]

                    if line_text:  # ignore empty lines
                        page_structure.append({
                            "text": line_text,
                            "bbox": bbox,
                            "size": size,
                            "font": font
                        })
        structure_data.append(page_structure)

    return structure_data

In [4]:
def compare_pages(page1: List[Dict], page2: List[Dict], threshold_px: float = 10.0) -> float:
    matches = 0
    total = max(len(page1), len(page2))

    for i in range(min(len(page1), len(page2))):
        el1 = page1[i]
        el2 = page2[i]

        # Text similarity
        text_sim = SequenceMatcher(None, el1['text'], el2['text']).ratio()

        # Position and style similarity
        bbox_diff = sum(abs(a - b) for a, b in zip(el1['bbox'], el2['bbox']))
        size_diff = abs(el1['size'] - el2['size']) if el1['size'] and el2['size'] else 0
        font_match = el1['font'] == el2['font']

        if text_sim > 0.85 and bbox_diff < threshold_px and size_diff < 2 and font_match:
            matches += 1

    return (matches / total * 100) if total else 0

In [5]:
def compare_documents(doc1_path: str, doc2_path: str, threshold_px: float = 10.0):
    struct1 = extract_structure(doc1_path)
    struct2 = extract_structure(doc2_path)

    min_pages = min(len(struct1), len(struct2))
    match_scores = []

    for i in range(min_pages):
        match_percent = compare_pages(struct1[i], struct2[i], threshold_px)
        match_scores.append(match_percent)
        print(f"Page {i + 1}: {match_percent:.2f}% match")

    overall = sum(match_scores) / len(match_scores) if match_scores else 0
    print(f"\n🔍 Overall Document Match: {overall:.2f}%")

    return overall

In [19]:
if __name__ == "__main__":
    # Provide your PDF paths here
    doc1_path = "/content/drive/MyDrive/Colab Notebooks/mypdf2-1-3.pdf"
    doc2_path = "/content/drive/MyDrive/Colab Notebooks/mypdf2-1-3_New_Test.pdf"

    # Optional: change threshold if needed
    threshold_px = 15.0  # max allowed bbox difference

    compare_documents(doc1_path, doc2_path, threshold_px)

Page 1: 75.00% match
Page 2: 93.48% match
Page 3: 100.00% match

🔍 Overall Document Match: 89.49%
